# MTL QSAR Cardiotoxicity — External Validation Evaluation
## Regression | Multi-Task Learning (Task-Specific Backbone)
Evaluasi performa **7 konfigurasi model** × **3 endpoint** × **3 difficulty level**  
menggunakan Bootstrap CI, scatter plot, residual plot, dan top-50% low-error compounds.


In [ ]:
import os, sys, json, ast, copy, random, pickle, math, csv, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED        = 42
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PIN_MEMORY  = (DEVICE == "cuda")
NON_BLOCKING = True
KEY_COL      = "SMILES"
TARGET_COL   = "pIC50"
TOP_FRAC     = 0.50
ALL_ENDPOINTS = ["hERG", "Cav1.2", "Nav1.5"]
LEVELS        = ["easy", "medium", "hard"]
N_BOOTSTRAP   = 1000

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"[INFO] DEVICE = {DEVICE}")


In [ ]:
# =============================================================================
# GLOBAL CONFIG — sesuaikan path sesuai environment
# =============================================================================

META_MD_COLS = [
    "SMILES", "InChIKey", "Compound_ID", "CompoundID",
    "Activity", "Activity_Label", "ActivityLabel", "pIC50",
    "Source", "Source.1",
    "Max_Tanimoto_to_Dev", "Max_Tanimoto_to_Dev.1",
    "MaxTanimototoDev", "Max_Tanimoto",
]

OUT_DIR = r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\MTL\Performance_Evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

SHARED_MD_COLS_PATH = r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\md_aligned_feature_cols.txt"

MTL_CONFIGS = {
    "FP": {
        "use_fp": True, "use_md": False, "use_g": False,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\MTL\FP\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\MTL\FP\final\target_scaler.pkl",
        "scaler_md_path":     None,
        "arch_type": "uniform",  "n_layers": 3, "hidden_dim": 511,
        "dropout": 0.012787584591142764,
    },
    "MD": {
        "use_fp": False, "use_md": True, "use_g": False,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\MTL\MD\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\MTL\MD\final\target_scaler.pkl",
        "scaler_md_path":     r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\MTL\MD\final\scaler_md.pkl",
        "arch_type": "wide_first", "n_layers": 3, "hidden_dim": 242,
        "dropout": 0.07051747878721783,
    },
    "Graph": {
        "use_fp": False, "use_md": False, "use_g": True,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\Graph\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\Graph\final\target_scaler.pkl",
        "scaler_md_path":     None,
        "arch_type": "uniform",  "n_layers": 3, "hidden_dim": 378,
        "dropout": 0.06721988785756863,
    },
    "FP+MD": {
        "use_fp": True, "use_md": True, "use_g": False,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+MD\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+MD\final\target_scaler.pkl",
        "scaler_md_path":     r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+MD\final\scaler_md.pkl",
        "arch_type": "wide_first", "n_layers": 3, "hidden_dim": 491,
        "dropout": 0.29086317263928513,
    },
    "FP+Graph": {
        "use_fp": True, "use_md": False, "use_g": True,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+Graph\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+Graph\final\target_scaler.pkl",
        "scaler_md_path":     None,
        "arch_type": "wide_first", "n_layers": 6, "hidden_dim": 433,
        "dropout": 0.058298055947197995,
    },
    "MD+Graph": {
        "use_fp": False, "use_md": True, "use_g": True,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\MD+Graph\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\MD+Graph\final\target_scaler.pkl",
        "scaler_md_path":     r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\MD+Graph\final\scaler_md.pkl",
        "arch_type": "wide_first", "n_layers": 4, "hidden_dim": 285,
        "dropout": 0.16413922468079553,
    },
    "FP+MD+Graph": {
        "use_fp": True, "use_md": True, "use_g": True,
        "model_path":         r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+MD+Graph\final\final_model.pt",
        "target_scaler_path": r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+MD+Graph\final\target_scaler.pkl",
        "scaler_md_path":     r"D:\QSAR Cardiotoxicity\Regression_Multitask_REVISED\results\MTL\FP+MD+Graph\final\scaler_md.pkl",
        "arch_type": "wide_first", "n_layers": 2, "hidden_dim": 292,
        "dropout": 0.1506182293413755,
    },
}

EXTERNAL_SETS = {
    "hERG": {
        "easy":   {"fp": r"D:\QSAR Cardiotoxicity\Regression Training\hERG\Descriptor\hERG_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression Training\hERG\Descriptor\hERG_ExternalVal1_easy_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\hERG\MD_Aligned_to_DevSet\hERG_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"},
        "medium": {"fp": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\hERG_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\hERG_ExternalVal2_medium_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\hERG\MD_Aligned_to_DevSet\hERG_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"},
        "hard":   {"fp": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\hERG_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\hERG_ExternalVal3_hard_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\hERG\MD_Aligned_to_DevSet\hERG_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"},
    },
    "Cav1.2": {
        "easy":   {"fp": r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\Descriptor\Cav1.2_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\Descriptor\Cav1.2_ExternalVal1_easy_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"},
        "medium": {"fp": r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\Descriptor\Cav1.2_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Cav1.2_ExternalVal2_medium_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"},
        "hard":   {"fp": r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\Descriptor\Cav1.2_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Cav1.2_ExternalVal3_hard_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\Cav1.2\MD_Aligned_to_DevSet\Cav1.2_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"},
    },
    "Nav1.5": {
        "easy":   {"fp": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\Nav1.5_ExternalVal1_easy_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Nav1.5_ExternalVal1_easy_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_easy_RDKit_CDK_aligned_exact_to_dev.xlsx"},
        "medium": {"fp": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\Nav1.5_ExternalVal2_medium_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Nav1.5_ExternalVal2_medium_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_medium_RDKit_CDK_aligned_exact_to_dev.xlsx"},
        "hard":   {"fp": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\FP\Nav1.5_ExternalVal3_hard_with_ECFP4_MACCS_APF.csv",
                   "graph": r"D:\QSAR Cardiotoxicity\Regression_Descriptor Calculation\Graph\Nav1.5_ExternalVal3_hard_graphs_67feat_with_pIC50.pkl",
                   "md":    r"D:\QSAR Cardiotoxicity\Regression Training\Nav1.5\MD_Aligned_to_DevSet\Nav1.5_hard_RDKit_CDK_aligned_exact_to_dev.xlsx"},
    },
}

print(f"[INFO] OUT_DIR = {OUT_DIR}")
print(f"[INFO] Configs : {list(MTL_CONFIGS.keys())}")


## Section 1 — Utility Functions

In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def fast_parse_list(s):
    if isinstance(s, list):       return s
    if isinstance(s, np.ndarray): return s.tolist()
    if not isinstance(s, str):    raise TypeError(f"Unexpected type: {type(s)}")
    try:    return json.loads(s)
    except: return ast.literal_eval(s)

def compute_fp_matrix_from_df(df_fp, ec_col="ECFP4", mc_col="MACCS"):
    ec = df_fp[ec_col].tolist(); mc = df_fp[mc_col].tolist()
    ec0 = fast_parse_list(ec[0]); mc0 = fast_parse_list(mc[0])
    d_ec, d_mc = len(ec0), len(mc0)
    X = np.zeros((len(df_fp), d_ec + d_mc), dtype=np.float32)
    for i in range(len(df_fp)):
        e = np.asarray(fast_parse_list(ec[i]), dtype=np.float32)
        m = np.asarray(fast_parse_list(mc[i]), dtype=np.float32)
        X[i, :d_ec] = e; X[i, d_ec:] = m
    return X, d_ec, d_mc

def get_mordred_feature_cols(df_md, meta_cols):
    return [c for c in df_md.columns
            if c not in meta_cols and pd.api.types.is_numeric_dtype(df_md[c])]

def load_shared_md_cols(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"md_aligned_feature_cols.txt tidak ada: {path}")
    cols = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("Strategy") and not line.startswith("N features"):
                cols.append(line)
    print(f"[INFO] Loaded {len(cols)} shared MD cols")
    return cols

# ---------- metrics ----------
def compute_all_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, np.float64).reshape(-1)
    mse    = float(mean_squared_error(y_true, y_pred))
    rmse   = float(np.sqrt(mse))
    mae    = float(mean_absolute_error(y_true, y_pred))
    denom  = np.maximum(np.abs(y_true), eps)
    mape   = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    r2     = float(r2_score(y_true, y_pred))
    r, p   = stats.pearsonr(y_true, y_pred) if len(y_true) > 1 else (np.nan, np.nan)
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape,
            "R2": r2, "Pearson_r": float(r), "N": len(y_true)}

def ape_per_row(y_true, y_pred, eps=1e-8):
    return np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), eps) * 100.0

def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape_score(y_true, y_pred, eps=1e-8):
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

# ---------- bootstrap CI ----------
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=N_BOOTSTRAP, ci=95, seed=SEED):
    """
    Kembalikan (median, lower, upper, half_width).
    half_width = (upper - lower) / 2  -> cocok untuk format "mean ± delta".
    """
    rng    = np.random.RandomState(seed)
    y_true = np.asarray(y_true, np.float64)
    y_pred = np.asarray(y_pred, np.float64)
    n      = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            pass
    if not scores:
        return np.nan, np.nan, np.nan, np.nan
    scores = np.array(scores)
    lo = float(np.percentile(scores, (100 - ci) / 2))
    hi = float(np.percentile(scores, 100 - (100 - ci) / 2))
    med   = float(np.median(scores))
    delta = (hi - lo) / 2.0
    return med, lo, hi, delta

# ---------- TargetScaler (identik dengan training notebook) ----------
class TargetScaler:
    def __init__(self):
        self.mean_ = {}; self.std_ = {}
    def fit(self, ep, y):
        self.mean_[ep] = float(np.nanmean(y))
        self.std_[ep]  = float(np.nanstd(y)) + 1e-8
    def transform(self, ep, y):
        return (np.asarray(y, np.float32) - self.mean_[ep]) / self.std_[ep]
    def inverse_transform(self, ep, y):
        return np.asarray(y, np.float32) * self.std_[ep] + self.mean_[ep]

def load_target_scaler(path):
    with open(path, "rb") as f:
        return pickle.load(f)

print("[OK] Utility functions loaded.")


## Section 2 — Data Loading

In [ ]:
# =============================================================================
# DATA LOADING — External Validation Set
# Mengembalikan dict: X_fp, X_md_raw, graphs, y, smiles, in_fp, in_md, in_g
# =============================================================================

try:
    from torch_geometric.data import Data, Batch
    from torch_geometric.nn import GCNConv
    HAS_PYG = True
except ImportError:
    Data = Batch = GCNConv = None
    HAS_PYG = False
    print("[WARN] torch_geometric tidak tersedia — Graph modality dinonaktifkan.")


def _load_fp(fp_path):
    df     = pd.read_csv(fp_path)
    X, _, _ = compute_fp_matrix_from_df(df)
    y      = df[TARGET_COL].astype(np.float32).to_numpy()
    smiles = df[KEY_COL].astype(str).tolist() if KEY_COL in df.columns else [str(i) for i in range(len(df))]
    return X, y, smiles


def _load_md(md_path, shared_cols):
    df = pd.read_excel(md_path)
    y  = df[TARGET_COL].astype(np.float32).to_numpy()
    smiles = df[KEY_COL].astype(str).tolist() if KEY_COL in df.columns else [str(i) for i in range(len(df))]
    X = np.zeros((len(df), len(shared_cols)), dtype=np.float32)
    for j, col in enumerate(shared_cols):
        if col in df.columns:
            X[:, j] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(np.float32)
    return X, y, smiles


def _load_graph(graph_path):
    if not HAS_PYG or not os.path.exists(graph_path):
        return None, None, None
    with open(graph_path, "rb") as f:
        g_obj = pickle.load(f)
    graphs, y_list, sm_list = [], [], []
    for item in g_obj:
        gd   = item["graph"]
        x_np = np.asarray(gd["node_features"], dtype=np.float32)
        ei   = np.asarray(gd["edge_index"],    dtype=np.int64)
        if ei.ndim == 2 and ei.shape[0] != 2: ei = ei.T
        graphs.append(Data(x=torch.from_numpy(x_np),
                            edge_index=torch.from_numpy(ei)))
        y_list.append(float(item.get(TARGET_COL, item.get("pIC50", 0.0))))
        sm_list.append(str(item.get("SMILES", "")))
    return graphs, np.array(y_list, np.float32), sm_list


def load_external_data(endpoint, level, shared_cols):
    """
    Load semua modality untuk satu endpoint+level,
    align by SMILES intersection, return dict siap pakai.
    """
    paths     = EXTERNAL_SETS[endpoint][level]
    fp_path   = paths.get("fp",    "")
    md_path   = paths.get("md",    "")
    graph_path= paths.get("graph", "")

    # load raw
    X_fp_raw,  y_fp,  sm_fp  = _load_fp(fp_path)  if os.path.exists(fp_path)    else (None, None, [])
    X_md_raw,  y_md,  sm_md  = _load_md(md_path, shared_cols) if os.path.exists(md_path) else (None, None, [])
    graphs_raw, y_g,  sm_g   = _load_graph(graph_path)

    # Kumpulkan smiles sets dari modality yang berhasil di-load
    sm_sets = []
    if sm_fp:  sm_sets.append(set(sm_fp))
    if sm_md:  sm_sets.append(set(sm_md))
    if sm_g:   sm_sets.append(set(sm_g))

    if not sm_sets:
        raise RuntimeError(f"Tidak ada data yang berhasil di-load untuk {endpoint}|{level}")

    common = sorted(list(set.intersection(*sm_sets)))
    if len(common) == 0:
        # fallback: gunakan FP smiles saja (tanpa alignment)
        print(f"  [WARN] {endpoint}|{level}: SMILES overlap=0, fallback ke FP-only alignment")
        common = sm_fp

    # Build index maps
    idx_fp = {s: i for i, s in enumerate(sm_fp)}  if sm_fp  else {}
    idx_md = {s: i for i, s in enumerate(sm_md)}  if sm_md  else {}
    idx_g  = {s: i for i, s in enumerate(sm_g)}   if sm_g   else {}

    n = len(common)
    idx_arr_fp = np.array([idx_fp[s] for s in common if s in idx_fp])
    idx_arr_md = np.array([idx_md[s] for s in common if s in idx_md])

    X_fp_out   = X_fp_raw[idx_arr_fp]   if X_fp_raw  is not None and len(idx_arr_fp) == n else X_fp_raw
    X_md_out   = X_md_raw[idx_arr_md]   if X_md_raw  is not None and len(idx_arr_md) == n else X_md_raw
    graphs_out = [graphs_raw[idx_g[s]] for s in common if s in idx_g] if graphs_raw else None

    # y — ambil dari FP (prioritas) lalu MD lalu Graph
    if X_fp_raw is not None and len(idx_arr_fp) == n:
        y_out = y_fp[idx_arr_fp]
    elif X_md_raw is not None and len(idx_arr_md) == n:
        y_out = y_md[idx_arr_md]
    else:
        y_out = y_g[np.array([idx_g[s] for s in common])] if y_g is not None else np.zeros(n, np.float32)

    in_fp = X_fp_out.shape[1] if X_fp_out is not None else 0
    in_md = X_md_out.shape[1] if X_md_out is not None else 0
    in_g  = None
    if graphs_out:
        g0 = next((g for g in graphs_out if g is not None), None)
        if g0 is not None: in_g = g0.x.shape[1]

    print(f"  [DATA] {endpoint}|{level}: N={n}, in_fp={in_fp}, in_md={in_md}, in_g={in_g}")
    return {
        "X_fp": X_fp_out, "X_md_raw": X_md_out,
        "graphs": graphs_out, "y": y_out, "smiles": common,
        "in_fp": in_fp, "in_md": in_md, "in_g": in_g,
    }

print("[OK] Data loading functions ready.")


## Section 3 — Model Architecture & Loading

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE (identik dengan training notebook — jangan diubah)
# =============================================================================

class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )
    def forward(self, x): return self.net(x)


class GraphEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=128, dropout=0.0):
        super().__init__()
        if not HAS_PYG: raise ImportError("torch_geometric not available")
        self.conv1   = GCNConv(in_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, batch):
        x, ei, bidx = batch.x, batch.edge_index, batch.batch
        h = F.relu(self.conv1(x, ei)); h = self.dropout(h)
        h = F.relu(self.conv2(h, ei))
        n_graphs = int(bidx.max().item()) + 1
        out = []
        for g in range(n_graphs):
            mask = (bidx == g)
            out.append(h[mask].mean(dim=0, keepdim=True))
        return torch.cat(out, dim=0)


def _build_backbone(concat_dim, hidden_dim, n_layers, arch_type, dropout):
    arch_type = arch_type or "uniform"
    n_layers  = max(int(n_layers), 1)
    if arch_type == "uniform":
        layer_dims = [concat_dim] + [hidden_dim] * n_layers
    elif arch_type == "pyramid":
        dims = [concat_dim]; cur = hidden_dim
        for _ in range(n_layers):
            dims.append(cur); cur = max(cur // 2, 32)
        layer_dims = dims
    elif arch_type == "wide_first":
        layer_dims = [concat_dim, hidden_dim * 2] + [hidden_dim] * max(n_layers - 1, 0)
    else:
        layer_dims = [concat_dim] + [hidden_dim] * n_layers
    layers = []
    for in_d, out_d in zip(layer_dims[:-1], layer_dims[1:]):
        layers += [nn.Linear(in_d, out_d), nn.ReLU(), nn.Dropout(dropout)]
    return nn.Sequential(*layers), layer_dims[-1]


class MTLModularNetReg(nn.Module):
    """
    REVISED ARCHITECTURE:
      Shared Encoder (FP / MD / Graph)
          |
    BB_hERG  BB_Cav1_2  BB_Nav1_5   <- task-specific backbone
       |          |          |
    Head       Head       Head       <- output pIC50
    """
    def __init__(self, in_fp=0, in_md=0, in_g=None,
                 use_fp=True, use_md=True, use_g=False,
                 hidden_dim=128, dropout=0.2,
                 n_layers=2, arch_type="uniform",
                 task_names=None):
        super().__init__()
        self.task_names      = task_names or ALL_ENDPOINTS
        self.safe_task_names = [t.replace(".", "_") for t in self.task_names]
        self.name_map        = dict(zip(self.task_names, self.safe_task_names))
        self.use_fp = use_fp and (in_fp > 0)
        self.use_md = use_md and (in_md > 0)
        self.use_g  = use_g  and (in_g is not None) and (in_g > 0)
        fp_out = md_out = g_out = 0
        if self.use_fp:
            self.fp_encoder = MLPEncoder(in_fp, hidden_dim, hidden_dim, dropout); fp_out = hidden_dim
        if self.use_md:
            self.md_encoder = MLPEncoder(in_md, hidden_dim, hidden_dim, dropout); md_out = hidden_dim
        if self.use_g:
            self.g_encoder  = GraphEncoder(in_g, 64, hidden_dim, dropout);        g_out  = hidden_dim
        concat_dim = fp_out + md_out + g_out
        assert concat_dim > 0
        self.task_backbones = nn.ModuleDict()
        self.task_heads     = nn.ModuleDict()
        for safe_t in self.safe_task_names:
            bb, bb_out = _build_backbone(concat_dim, hidden_dim, n_layers, arch_type, dropout)
            self.task_backbones[safe_t] = bb
            self.task_heads[safe_t]     = nn.Sequential(
                nn.Linear(bb_out, bb_out // 2), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(bb_out // 2, 1))

    def forward(self, x_fp=None, x_md=None, g_batch=None):
        feats = []
        if self.use_fp and x_fp    is not None: feats.append(self.fp_encoder(x_fp))
        if self.use_md and x_md    is not None: feats.append(self.md_encoder(x_md))
        if self.use_g  and g_batch is not None: feats.append(self.g_encoder(g_batch))
        assert feats
        z = feats[0] if len(feats) == 1 else torch.cat(feats, dim=1)
        return {orig: self.task_heads[safe_t](self.task_backbones[safe_t](z)).squeeze(1)
                for orig, safe_t in self.name_map.items()}


def build_and_load_model(cfg, in_fp, in_md, in_g):
    model = MTLModularNetReg(
        in_fp=in_fp, in_md=in_md, in_g=in_g,
        use_fp=cfg["use_fp"], use_md=cfg["use_md"], use_g=cfg["use_g"],
        hidden_dim=cfg["hidden_dim"], dropout=cfg["dropout"],
        n_layers=cfg["n_layers"], arch_type=cfg["arch_type"],
        task_names=ALL_ENDPOINTS,
    ).to(DEVICE)
    state = torch.load(cfg["model_path"], map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    n_p = sum(p.numel() for p in model.parameters())
    print(f"    Model loaded | params={n_p:,}")
    return model

print("[OK] Model architecture defined.")


## Section 4 — DataLoader & Evaluate

In [ ]:
# =============================================================================
# DATASET + DATALOADER + EVALUATE
# =============================================================================

class ExternalValDataset(Dataset):
    def __init__(self, X_fp, X_md, graphs, y):
        self.X_fp = X_fp; self.X_md = X_md
        self.graphs = graphs
        self.y = np.asarray(y, np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x_fp = torch.from_numpy(self.X_fp[idx]).float() if self.X_fp   is not None else None
        x_md = torch.from_numpy(self.X_md[idx]).float() if self.X_md   is not None else None
        g    = self.graphs[idx]                          if self.graphs is not None else None
        return x_fp, x_md, g, torch.tensor(self.y[idx], dtype=torch.float32)


def collate_fn_eval(batch):
    x_fp_list, x_md_list, g_list, y_list = zip(*batch)
    has_fp = any(x is not None for x in x_fp_list)
    has_md = any(x is not None for x in x_md_list)
    has_g  = any(g is not None for g in g_list)
    x_fp_b = torch.stack(list(x_fp_list)) if has_fp else None
    x_md_b = torch.stack(list(x_md_list)) if has_md else None
    if HAS_PYG and has_g:
        valid  = [g for g in g_list if g is not None]
        g_b    = Batch.from_data_list(valid) if valid else None
    else:
        g_b = None
    return x_fp_b, x_md_b, g_b, torch.stack(list(y_list))


def make_loader(X_fp, X_md, graphs, y, batch_size=256):
    ds = ExternalValDataset(X_fp, X_md, graphs, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      collate_fn=collate_fn_eval)


def evaluate_mtl_endpoint(model, loader, endpoint, tsc):
    """
    Inference satu endpoint.
    y_b adalah y asli (original scale) — digunakan sebagai label aktual.
    Prediksi model di-inverse_transform dari scaled space.
    """
    model.eval()
    all_y, all_pred = [], []
    with torch.no_grad():
        for x_fp, x_md, g_b, y_b in loader:
            y_b = y_b.to(DEVICE, non_blocking=NON_BLOCKING)
            if x_fp is not None: x_fp = x_fp.to(DEVICE, non_blocking=NON_BLOCKING)
            if x_md is not None: x_md = x_md.to(DEVICE, non_blocking=NON_BLOCKING)
            if g_b  is not None: g_b  = g_b.to(DEVICE)
            pred_dict = model(x_fp=x_fp, x_md=x_md, g_batch=g_b)
            p_ep = pred_dict[endpoint].cpu().numpy()
            p_ep = tsc.inverse_transform(endpoint, p_ep)
            all_y.append(y_b.cpu().numpy())
            all_pred.append(p_ep)
    return (np.concatenate(all_y).astype(np.float64),
            np.concatenate(all_pred).astype(np.float64))

print("[OK] DataLoader & evaluate ready.")


## Section 5 — Plotting & Error Table

In [ ]:
# =============================================================================
# PLOTTING FUNCTIONS
# =============================================================================

def plot_actual_vs_predicted(config_name, endpoint, level, y_true, y_pred, out_dir):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_true, y_pred, alpha=0.6, s=18, edgecolors="none")
    lo = min(float(np.min(y_true)), float(np.min(y_pred)))
    hi = max(float(np.max(y_true)), float(np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.2)
    ax.set_xlabel("Actual pIC50"); ax.set_ylabel("Predicted pIC50")
    ax.set_title(f"MTL-{config_name} | {endpoint} | {level}")
    safe = config_name.replace("+", "_"); ep_safe = endpoint.replace(".", "_")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"MTL_{safe}_{ep_safe}_{level}_AvP.png"), dpi=150)
    plt.close(fig)


def plot_residuals(config_name, endpoint, level, y_true, y_pred, out_dir):
    residuals = y_true - y_pred
    fig, ax   = plt.subplots(figsize=(5, 4))
    ax.scatter(y_pred, residuals, alpha=0.6, s=18, edgecolors="none")
    ax.axhline(0.0, color="r", linestyle="--", lw=1.2)
    ax.set_xlabel("Predicted pIC50"); ax.set_ylabel("Residual (Actual − Predicted)")
    ax.set_title(f"MTL-{config_name} | {endpoint} | {level} — Residuals")
    safe = config_name.replace("+", "_"); ep_safe = endpoint.replace(".", "_")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"MTL_{safe}_{ep_safe}_{level}_Residuals.png"), dpi=150)
    plt.close(fig)


# =============================================================================
# ERROR TABLE HELPERS
# =============================================================================

def make_error_dataframe(smiles, y_true, y_pred, config_name, endpoint, level):
    y_true = np.asarray(y_true, np.float64)
    y_pred = np.asarray(y_pred, np.float64)
    df = pd.DataFrame({
        "Config":          config_name,
        "Endpoint":        endpoint,
        "Level":           level,
        "SMILES":          smiles,
        "Measured_pIC50":  y_true,
        "Predicted_pIC50": y_pred,
        "Residual":        y_true - y_pred,
        "Absolute_Error":  np.abs(y_true - y_pred),
        "Squared_Error":   (y_true - y_pred) ** 2,
        "APE_percent":     ape_per_row(y_true, y_pred),
    })
    df = df.sort_values("Absolute_Error", ascending=True).reset_index(drop=True)
    df["Rank_by_Smallest_AE"] = np.arange(1, len(df) + 1)
    df["Cumulative_Fraction"]  = df["Rank_by_Smallest_AE"] / len(df)
    return df


def select_top_small_error(df, frac=TOP_FRAC):
    n_keep = max(1, int(math.ceil(len(df) * frac)))
    return df.sort_values("Absolute_Error").head(n_keep).reset_index(drop=True)

print("[OK] Plotting & error table functions ready.")


## Section 6 — Initialise Output

In [ ]:
# =============================================================================
# LOAD SHARED_MD_COLS & INITIALISE CSV
# =============================================================================

SHARED_MD_COLS = load_shared_md_cols(SHARED_MD_COLS_PATH)
print(f"[INFO] MD feature cols ready: {len(SHARED_MD_COLS)} cols")
print(f"[INFO] First 5: {SHARED_MD_COLS[:5]}")

CSV_PATH   = os.path.join(OUT_DIR, "MTL_external_regression_performance.csv")
CSV_HEADER = [
    "Config", "Endpoint", "Level",
    "RMSE", "RMSE_med", "RMSE_CI_lower", "RMSE_CI_upper", "RMSE_delta", "RMSE_display",
    "MAE",  "MAE_med",  "MAE_CI_lower",  "MAE_CI_upper",  "MAE_delta",  "MAE_display",
    "R2",   "R2_med",   "R2_CI_lower",   "R2_CI_upper",   "R2_delta",   "R2_display",
    "MAPE", "MAPE_med", "MAPE_CI_lower", "MAPE_CI_upper", "MAPE_delta", "MAPE_display",
    "Pearson_r", "N",
]
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(CSV_HEADER)

error_tables = {}
print(f"[OK] CSV initialised: {CSV_PATH}")


## Section 7 — Main Evaluation Loop

In [ ]:
# =============================================================================
# MAIN EVALUATION LOOP
# Urutan operasi per kombinasi (endpoint x level x config):
#   1. Load scaler_md  (sebelum data, agar bisa cek shape)
#   2. Load external data  (FP / MD / Graph, align by SMILES)
#   3. Scale MD dengan scaler_md
#   4. Build & load model
#   5. Evaluate -> y_true, y_pred
#   6. Metrics + Bootstrap CI
#   7. Plot: actual-vs-predicted + residuals
#   8. Build error table
#   9. Append CSV row
# =============================================================================

SEP = "=" * 65

for endpoint in ALL_ENDPOINTS:
    for level in LEVELS:
        print("\n" + SEP)
        print(f"  Endpoint: {endpoint}  |  Level: {level}")
        print(SEP)

        for config_name, cfg in MTL_CONFIGS.items():
            print(f"\n  -- Config: {config_name} --")

            # -- 1. Load scaler_md --------------------------------------------
            scaler_md = None
            if cfg["use_md"] and cfg["scaler_md_path"]:
                try:
                    with open(cfg["scaler_md_path"], "rb") as f:
                        scaler_md = pickle.load(f)
                    n_feat = getattr(scaler_md, "n_features_in_", "?")
                    print(f"    scaler_md loaded | n_features_in_={n_feat}")
                except Exception as e:
                    print(f"    [ERROR] Gagal load scaler_md: {e}")
                    continue

            # -- 2. Load data -------------------------------------------------
            try:
                data = load_external_data(endpoint, level, shared_cols=SHARED_MD_COLS)
            except Exception as e:
                print(f"    [ERROR] Data loading failed: {e}")
                continue

            X_fp_raw = data["X_fp"]
            X_md_raw = data["X_md_raw"]
            graphs   = data["graphs"]
            y_true   = data["y"]
            smiles   = data["smiles"]
            in_fp    = data["in_fp"]
            in_g     = data["in_g"]
            in_md    = data["in_md"]

            if cfg["use_md"] and X_md_raw is None:
                print("    [SKIP] MD required but not available")
                continue

            # -- 3. Load target scaler ----------------------------------------
            try:
                tsc = load_target_scaler(cfg["target_scaler_path"])
            except Exception as e:
                print(f"    [ERROR] Cannot load target scaler: {e}")
                continue

            if endpoint not in tsc.mean_:
                print(f"    [SKIP] Endpoint '{endpoint}' tidak ada di scaler: "
                      f"{list(tsc.mean_.keys())}")
                continue

            # -- 4. Scale MD --------------------------------------------------
            X_md_scaled = None
            if cfg["use_md"] and X_md_raw is not None:
                if scaler_md is not None:
                    try:
                        expected = getattr(scaler_md, "n_features_in_", in_md)
                        assert X_md_raw.shape[1] == expected, (
                            f"Shape mismatch! X_md={X_md_raw.shape[1]} "
                            f"vs scaler={expected}")
                        X_md_scaled = scaler_md.transform(X_md_raw).astype(np.float32)
                        print(f"    MD scaled: {X_md_scaled.shape}")
                    except Exception as e:
                        print(f"    [ERROR] MD scaling failed: {e}")
                        continue
                else:
                    X_md_scaled = X_md_raw

            # -- 5. Siapkan input sesuai modalitas aktif ----------------------
            X_fp_in = X_fp_raw if cfg["use_fp"] else None
            g_in    = graphs   if cfg["use_g"]  else None
            in_fp_m = in_fp    if cfg["use_fp"] else 0
            in_g_m  = in_g     if cfg["use_g"]  else None
            in_md_m = in_md    if cfg["use_md"] else 0

            # -- 6. Build & load model ----------------------------------------
            try:
                model = build_and_load_model(cfg, in_fp=in_fp_m, in_md=in_md_m, in_g=in_g_m)
            except Exception as e:
                print(f"    [ERROR] Model load failed: {e}")
                continue

            # -- 7. DataLoader ------------------------------------------------
            loader = make_loader(X_fp_in, X_md_scaled, g_in, y_true, batch_size=256)

            # -- 8. Evaluate --------------------------------------------------
            try:
                y_eval, pred_eval = evaluate_mtl_endpoint(model, loader, endpoint, tsc)
            except Exception as e:
                print(f"    [ERROR] Evaluation failed: {e}")
                continue

            # -- 9. Metrics + Bootstrap CI ------------------------------------
            metrics = compute_all_metrics(y_eval, pred_eval)

            rmse_med, rmse_lo, rmse_hi, rmse_d = bootstrap_ci(y_eval, pred_eval, rmse_score)
            mae_med,  mae_lo,  mae_hi,  mae_d  = bootstrap_ci(y_eval, pred_eval, mean_absolute_error)
            r2_med,   r2_lo,   r2_hi,   r2_d   = bootstrap_ci(y_eval, pred_eval, r2_score)
            mape_med, mape_lo, mape_hi, mape_d = bootstrap_ci(y_eval, pred_eval, mape_score)

            msg = (f"    RMSE={metrics['RMSE']:.4f} ({rmse_med:.4f}\u00b1{rmse_d:.4f})  "
                   f"MAE={metrics['MAE']:.4f} ({mae_med:.4f}\u00b1{mae_d:.4f})  "
                   f"R2={metrics['R2']:.4f} ({r2_med:.4f}\u00b1{r2_d:.4f})  "
                   f"MAPE={metrics['MAPE']:.2f}%  N={metrics['N']}")
            print(msg)

            # -- 10. Plots ----------------------------------------------------
            plot_actual_vs_predicted(config_name, endpoint, level, y_eval, pred_eval, OUT_DIR)
            plot_residuals(config_name, endpoint, level, y_eval, pred_eval, OUT_DIR)

            # -- 11. Error table ----------------------------------------------
            df_err = make_error_dataframe(
                smiles, y_eval, pred_eval, config_name, endpoint, level)
            error_tables[(config_name, endpoint, level)] = df_err

            # -- 12. Append CSV -----------------------------------------------
            with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow([
                    config_name, endpoint, level,
                    metrics["RMSE"], rmse_med, rmse_lo, rmse_hi, rmse_d,
                    f"{rmse_med:.4f} \u00b1 {rmse_d:.4f}",
                    metrics["MAE"],  mae_med,  mae_lo,  mae_hi,  mae_d,
                    f"{mae_med:.4f} \u00b1 {mae_d:.4f}",
                    metrics["R2"],   r2_med,   r2_lo,   r2_hi,   r2_d,
                    f"{r2_med:.4f} \u00b1 {r2_d:.4f}",
                    metrics["MAPE"], mape_med, mape_lo, mape_hi, mape_d,
                    f"{mape_med:.4f} \u00b1 {mape_d:.4f}",
                    metrics["Pearson_r"], metrics["N"],
                ])

            del model
            torch.cuda.empty_cache()

print(f"\n[DONE] Evaluation loop selesai. {len(error_tables)} kombinasi berhasil.")


## Section 8 — Save Top-50% Low-Error Compounds (Excel)

In [ ]:
# =============================================================================
# SAVE TOP-50% SMALL-ERROR COMPOUNDS → EXCEL
# =============================================================================

EXCEL_PATH = os.path.join(OUT_DIR, "MTL_top50pct_small_error_compounds.xlsx")

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    summary_rows = []
    for (config_name, endpoint, level), df_level in error_tables.items():
        df_top = select_top_small_error(df_level, frac=TOP_FRAC)
        sheet  = (
            f"{config_name}_{endpoint}_{level}"
            .replace("+", "_").replace(".", "_")
        )[:31]
        df_top.to_excel(writer, sheet_name=sheet, index=False)
        summary_rows.append({
            "Config":            config_name,
            "Endpoint":          endpoint,
            "Level":             level,
            "Total_Compounds":   len(df_level),
            "Top_Fraction_Kept": TOP_FRAC,
            "Rows_Kept":         len(df_top),
            "Mean_AE_in_Kept":   round(df_top["Absolute_Error"].mean(), 4),
            "Max_AE_in_Kept":    round(df_top["Absolute_Error"].max(),  4),
        })
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVE] Excel saved : {EXCEL_PATH}")


## Section 9 — Summary Report

In [ ]:
# =============================================================================
# SUMMARY REPORT -- Baca CSV, cetak tabel ringkas per endpoint
# =============================================================================

SEP70 = "=" * 70

df_csv = pd.read_csv(CSV_PATH)

for ep in ALL_ENDPOINTS:
    print("\n" + SEP70)
    print(f"  Endpoint: {ep}")
    print(SEP70)
    df_ep = df_csv[df_csv["Endpoint"] == ep]
    print(df_ep[["Config", "Level", "RMSE_display", "MAE_display",
                  "R2_display", "MAPE_display", "Pearson_r", "N"]].to_string(index=False))

print("\n" + SEP70)
print("  OUTPUT FILES")
print(SEP70)
print(f"  CSV metrics    : {CSV_PATH}")
print(f"  Excel top-50%  : {EXCEL_PATH}")
print(f"  Plots folder   : {OUT_DIR}")
